# Silver NLP layer v5 — Generic NL/EN suggestions only

This notebook reads the output from `silver_local_llm_v5.ipynb` and creates a generic local NLP suggestion layer.

Design principles:
- No document-specific hardcoded terms, people, organizations, places, or project names.
- Works for Dutch and English documents.
- Runs locally and does not call Ollama.
- Output can be used as hints by Gold and Gold Meta.
- Compatible with later larger Ollama models because the output format stays stable.
- Entities and keywords are suggestions only, not final metadata.


In [53]:

from pathlib import Path
import json
import re
import math
import unicodedata
from collections import Counter, defaultdict
from datetime import datetime

# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------
SILVER_DIR = Path("../../Data/silver")
OUT_DIR = Path("../../Data/silver_nlp")
OUT_DIR.mkdir(parents=True, exist_ok=True)

DOCUMENT_ID = "doc_01"
SILVER_JSON_PATH = SILVER_DIR / f"{DOCUMENT_ID}_silver.json"
CHUNKS_JSONL_PATH = SILVER_DIR / f"{DOCUMENT_ID}_chunks.jsonl"

if not SILVER_JSON_PATH.exists():
    raise FileNotFoundError(f"Silver JSON not found: {SILVER_JSON_PATH}")
if not CHUNKS_JSONL_PATH.exists():
    raise FileNotFoundError(f"Silver chunks JSONL not found: {CHUNKS_JSONL_PATH}")

with open(SILVER_JSON_PATH, "r", encoding="utf-8") as f:
    silver = json.load(f)

chunks = []
with open(CHUNKS_JSONL_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            chunks.append(json.loads(line))

main_text = silver.get("document_parts", {}).get("main_text", "")
sections = silver.get("sections", [])
detected_language = silver.get("detected_language", "unknown")
titlepage_candidates = silver.get("titlepage_candidates", {})

print("document_id:", DOCUMENT_ID)
print("detected_language:", detected_language)
print("main_text_words:", len(main_text.split()))
print("sections:", len(sections))
print("chunks:", len(chunks))


document_id: doc_01
detected_language: en
main_text_words: 13304
sections: 57
chunks: 57


In [54]:

# ---------------------------------------------------------------------
# Optional spaCy loading
# ---------------------------------------------------------------------
# This layer must remain local and model-independent. spaCy is optional.
# If the language model is unavailable, the notebook falls back to rule-based NLP.

SPACY_AVAILABLE = False
nlp = None
spacy_model_name = None

try:
    import spacy
    if detected_language == "nl":
        try_models = ["nl_core_news_sm", "xx_ent_wiki_sm"]
    elif detected_language == "en":
        try_models = ["en_core_web_sm", "xx_ent_wiki_sm"]
    else:
        try_models = ["xx_ent_wiki_sm", "en_core_web_sm", "nl_core_news_sm"]

    for model_name in try_models:
        try:
            nlp = spacy.load(model_name)
            spacy_model_name = model_name
            SPACY_AVAILABLE = True
            break
        except Exception:
            pass
except Exception:
    SPACY_AVAILABLE = False

print("spaCy available:", SPACY_AVAILABLE)
print("spaCy model:", spacy_model_name)


spaCy available: True
spaCy model: en_core_web_sm


In [55]:

# ---------------------------------------------------------------------
# Generic normalization and filters
# ---------------------------------------------------------------------
def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", text or "")
    text = text.replace("\u00a0", " ")
    text = re.sub(r"[\t\r]+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def simple_tokens(text: str):
    # Keeps Dutch/English letters, numbers and common technical separators.
    return re.findall(r"[A-Za-zÀ-ÖØ-öø-ÿ0-9][A-Za-zÀ-ÖØ-öø-ÿ0-9_+./#-]*", text or "")

GENERIC_STOPWORDS_NL = {
    "de", "het", "een", "en", "of", "dat", "dit", "die", "deze", "voor", "van", "in", "op", "aan", "om", "met", "als", "bij",
    "is", "zijn", "wordt", "worden", "werd", "werden", "heeft", "hebben", "had", "hadden", "kan", "kunnen", "zal", "zullen",
    "er", "ook", "nog", "dan", "maar", "wel", "niet", "geen", "meer", "minder", "door", "naar", "uit", "over", "onder",
    "tussen", "binnen", "buiten", "zoals", "waarin", "waarbij", "waarvoor", "waarna", "wanneer", "omdat", "daarnaast",
    "echter", "tevens", "vervolgens", "hierbij", "hierdoor", "daarom", "dus", "per", "tot", "toe", "af", "mee", "zelf",
    "hoofdstuk", "paragraaf", "figuur", "tabel", "bijlage", "noot", "pagina", "deelvraag", "vraag", "vragen", "antwoord",
    "gemaakt", "gebruikt", "gebruik", "maken", "komen", "gaan", "geven", "zien", "vinden", "beschreven", "genoemd",
    "verschillende", "andere", "aantal", "eerste", "tweede", "derde", "volgende", "mogelijk", "waarschijnlijk", "volledig"
}

GENERIC_STOPWORDS_EN = {
    "the", "a", "an", "and", "or", "that", "this", "these", "those", "for", "of", "in", "on", "to", "with", "as", "by",
    "is", "are", "was", "were", "be", "been", "being", "has", "have", "had", "can", "could", "will", "would", "should",
    "there", "also", "then", "but", "not", "no", "more", "less", "from", "about", "under", "between", "within", "without",
    "because", "therefore", "however", "thus", "per", "until", "chapter", "section", "figure", "table", "appendix", "note", "page",
    "question", "answer", "used", "using", "made", "make", "different", "other", "several", "first", "second", "third", "next",
    "possible", "probably", "full", "complete", "following", "described", "called", "mentioned"
}

STOPWORDS = GENERIC_STOPWORDS_NL | GENERIC_STOPWORDS_EN

BAD_SINGLE_TOKENS = {
    "x", "xx", "xxx", "max", "min", "must", "should", "could", "won", "have", "fig", "p", "pp", "et", "al",
    "jpg", "jpeg", "png", "pdf", "html", "txt", "docx", "zip"
}

SECTION_NUMBER_RE = re.compile(r"^\d+(?:\.\d+){0,5}$")
DECIMAL_RE = re.compile(r"^\d+[,.]\d+$")
NUMBER_ONLY_RE = re.compile(r"^[\d.,:/%-]+$")

DATE_PATTERNS = [
    re.compile(r"\b\d{1,2}[-/]\d{1,2}[-/]\d{2,4}\b"),
    re.compile(r"\b\d{4}[-/]\d{1,2}[-/]\d{1,2}\b"),
    re.compile(r"\b\d{1,2}\s+(?:januari|februari|maart|april|mei|juni|juli|augustus|september|oktober|november|december)\s+\d{4}\b", re.I),
    re.compile(r"\b(?:january|february|march|april|may|june|july|august|september|october|november|december)\s+\d{1,2},?\s+\d{4}\b", re.I),
    re.compile(r"\b(?:jan|feb|mar|apr|may|jun|jul|aug|sep|sept|oct|nov|dec)[a-z]*\s+\d{4}\b", re.I),
    re.compile(r"\b(?:januari|februari|maart|april|mei|juni|juli|augustus|september|oktober|november|december)\s+\d{4}\b", re.I),
]

def clean_term(term: str) -> str:
    term = normalize_text(term)
    term = re.sub(r"^[\W_]+|[\W_]+$", "", term)
    term = re.sub(r"\s+", " ", term)
    return term.strip()

def is_probable_date(text: str) -> bool:
    t = clean_term(text)
    if not t:
        return False
    if SECTION_NUMBER_RE.match(t) or DECIMAL_RE.match(t):
        return False
    if re.fullmatch(r"\d{4}", t):
        year = int(t)
        return 1500 <= year <= 2100
    return any(p.search(t) for p in DATE_PATTERNS)

def is_bad_term(term: str) -> bool:
    t = clean_term(term)
    tl = t.lower()
    if not t or len(t) < 2:
        return True
    if SECTION_NUMBER_RE.match(t) or NUMBER_ONLY_RE.match(t):
        return True
    if tl in STOPWORDS or tl in BAD_SINGLE_TOKENS:
        return True
    if len(tl) == 1:
        return True
    # Reject strings made mostly of table artifacts or punctuation.
    alpha_count = sum(ch.isalpha() for ch in t)
    if alpha_count < max(2, len(t) * 0.35):
        return True
    return False

def is_good_keyword_phrase(term: str) -> bool:
    term = clean_term(term)
    if is_bad_term(term):
        return False
    words = term.split()
    lowered = [w.lower().strip(".,;:()[]{}'") for w in words]
    if any(is_bad_term(w) for w in lowered if len(words) == 1):
        return False
    if lowered[0] in STOPWORDS or lowered[-1] in STOPWORDS:
        return False
    content_words = [w for w in lowered if w not in STOPWORDS and not is_bad_term(w)]
    if len(content_words) == 0:
        return False
    if len(words) >= 2 and len(content_words) < 1:
        return False
    if len(words) > 5:
        return False
    return True

def canonical_display(term: str) -> str:
    # Generic display normalization only. No project/document-specific mapping.
    t = clean_term(term)
    if re.fullmatch(r"[A-Za-z]{2,8}", t) and t.isupper():
        return t
    if t.lower() in {"ai", "nlp", "pdf", "json", "api", "cpu", "gpu", "ram", "sql", "http", "https"}:
        return t.upper()
    return t


In [56]:

# ---------------------------------------------------------------------
# Generic keyword extraction
# ---------------------------------------------------------------------
def extract_candidate_ngrams(text: str, min_n=1, max_n=4) -> Counter:
    tokens = simple_tokens(text)
    cleaned = []
    for tok in tokens:
        tok = tok.strip("'’-. ")
        if tok:
            cleaned.append(tok)

    counts = Counter()
    for n in range(min_n, max_n + 1):
        for i in range(len(cleaned) - n + 1):
            phrase = " ".join(cleaned[i:i+n])
            phrase = clean_term(phrase)
            if not is_good_keyword_phrase(phrase):
                continue
            counts[phrase.lower()] += 1
    return counts

def context_for_term(term: str, text: str, window: int = 160) -> str:
    if not term:
        return ""
    pattern = re.compile(re.escape(term), re.I)
    match = pattern.search(text)
    if not match:
        return ""
    start = max(0, match.start() - window)
    end = min(len(text), match.end() + window)
    snippet = text[start:end]
    snippet = re.sub(r"\s+", " ", snippet).strip()
    return snippet

def collapse_keyword_variants(items):
    # Generic: prefer longer multi-word terms over contained single-word terms.
    selected = []
    for item in items:
        low = item["term"].lower()
        words = set(low.split())
        contained = False
        for s in selected:
            slow = s["term"].lower()
            if low == slow:
                contained = True
                break
            if len(low.split()) == 1 and low in slow.split():
                contained = True
                break
            if len(words) > 1 and low in slow and item["score"] < s["score"] * 0.65:
                contained = True
                break
        if not contained:
            selected.append(item)
    return selected

def build_keyword_suggestions(sections, main_text: str, top_n=40):
    global_counts = Counter()
    section_hits = defaultdict(set)

    for sec in sections:
        sid = sec.get("section_id", "")
        text = sec.get("text", "")
        counts = extract_candidate_ngrams(text, 1, 4)
        for term, freq in counts.items():
            global_counts[term] += freq
            section_hits[term].add(sid)

    section_count = max(len(sections), 1)
    raw_items = []
    for term, freq in global_counts.items():
        if freq < 2:
            continue
        spread = len(section_hits[term])
        word_len = len(term.split())
        # Prefer meaningful multi-word terms, but do not hardcode topics.
        length_bonus = 1.0 + min(word_len - 1, 3) * 0.35
        spread_bonus = 1.0 + min(spread / section_count, 0.6)
        score = freq * length_bonus * spread_bonus
        display = canonical_display(term)
        raw_items.append({
            "term": display,
            "score": round(score, 3),
            "frequency": freq,
            "section_spread": spread,
            "section_spread_ratio": round(spread / section_count, 3),
            "context_example": context_for_term(term, main_text),
            "note": "generic_keyword_suggestion_based_on_frequency_spread_and_context"
        })

    raw_items.sort(key=lambda x: (x["score"], len(x["term"].split()), x["frequency"]), reverse=True)
    collapsed = collapse_keyword_variants(raw_items)

    # Final rank.
    out = []
    seen = set()
    for item in collapsed:
        low = item["term"].lower()
        if low in seen:
            continue
        seen.add(low)
        item["rank"] = len(out) + 1
        out.append(item)
        if len(out) >= top_n:
            break
    return out

keyword_suggestions = build_keyword_suggestions(sections, main_text, top_n=40)
print("keyword suggestions:", len(keyword_suggestions))
for k in keyword_suggestions[:10]:
    print(k["rank"], k["term"], k["frequency"], k["score"])


keyword suggestions: 40
1 data 221 353.6
2 model 117 180.632
3 eeg 95 143.333
4 it 80 128.0
5 seizure 77 116.175
6 seizures 75 105.263
7 graduation thesis angélique huige 31 96.997
8 research 68 95.439
9 which 54 82.421
10 graduation thesis angélique 31 80.437


In [57]:

# ---------------------------------------------------------------------
# Generic entity extraction
# ---------------------------------------------------------------------
ENTITY_CATEGORIES = [
    "people",
    "organizations",
    "locations",
    "dates",
    "methods_tools_or_products",
    "documents_or_standards",
    "events"
]

SPACY_LABEL_MAP = {
    "PERSON": "people",
    "PER": "people",
    "ORG": "organizations",
    "GPE": "locations",
    "LOC": "locations",
    "DATE": "dates",
    "EVENT": "events",
    "WORK_OF_ART": "documents_or_standards",
    "LAW": "documents_or_standards",
    "PRODUCT": "methods_tools_or_products",
    "NORP": "groups_or_nationalities",
}

ORG_SUFFIX_RE = re.compile(
    r"\b([A-ZÀ-Ö][\wÀ-ÖØ-öø-ÿ&.'-]*(?:\s+[A-ZÀ-Ö][\wÀ-ÖØ-öø-ÿ&.'-]*){0,5}\s+"
    r"(?:University|Universiteit|Hogeschool|College|Institute|Instituut|Agency|Bureau|Consortium|Foundation|Stichting|"
    r"Corporation|Corp|Company|Ltd|LLC|Inc|Group|GmbH|BV|NV|Ministerie|Gemeente|Provincie|Department|Lab|Laboratory|Labs))\b"
)

CAPITALIZED_SEQUENCE_RE = re.compile(
    r"\b[A-ZÀ-Ö][A-Za-zÀ-ÖØ-öø-ÿ'’-]+(?:\s+(?:van|de|der|den|het|the|of|for|and|&|en|op|von|du|del|da|dos|das))?"
    r"(?:\s+[A-ZÀ-Ö][A-Za-zÀ-ÖØ-öø-ÿ'’-]+){1,5}\b"
)

ACRONYM_RE = re.compile(r"\b[A-Z][A-Z0-9&/+.-]{1,9}\b")
TOOL_CONTEXT_RE = re.compile(r"\b(model|method|methode|tool|framework|library|bibliotheek|software|platform|standard|standaard|protocol|formula|formule|algorithm|algoritme|dataset|database|api|service|calculator|pipeline|system|systeem)\b", re.I)
DOC_CONTEXT_RE = re.compile(r"\b(report|rapport|paper|article|artikel|standard|standaard|richtlijn|policy|beleid|wet|law|dataset|onderzoek|study|studie|thesis|scriptie|portfolio)\b", re.I)
EVENT_CONTEXT_RE = re.compile(r"\b(meeting|workshop|conference|conferentie|interview|sprint|review|demo|presentation|presentatie)\b", re.I)

PERSON_CONTEXT_RE = re.compile(r"\b(author|auteur|written by|geschreven door|interview with|interview met|door|professor|dr\.?|ir\.?|mr\.?|ms\.?|mrs\.?|researcher|onderzoeker|student|advisor|adviseur|begeleider|lector)\b", re.I)
ORG_CONTEXT_RE = re.compile(r"\b(organization|organisation|organisatie|company|bedrijf|university|universiteit|hogeschool|institute|instituut|department|afdeling|team|ministry|ministerie|municipality|gemeente|province|provincie|consortium|foundation|stichting)\b", re.I)
LOCATION_CONTEXT_RE = re.compile(r"\b(city|stad|town|dorp|municipality|gemeente|province|provincie|country|land|region|regio|area|gebied|locatie|location)\b", re.I)

def nearby_context(text: str, start: int, end: int, window: int = 80) -> str:
    return text[max(0, start-window):min(len(text), end+window)]

def classify_capitalized_entity(candidate: str, context: str) -> str | None:
    c = clean_term(candidate)
    if is_bad_entity(c):
        return None
    if is_probable_date(c):
        return "dates"
    if ORG_CONTEXT_RE.search(context) or ORG_SUFFIX_RE.search(c):
        return "organizations"
    if TOOL_CONTEXT_RE.search(context) or c.isupper():
        return "methods_tools_or_products"
    if DOC_CONTEXT_RE.search(context):
        return "documents_or_standards"
    if EVENT_CONTEXT_RE.search(context):
        return "events"
    if LOCATION_CONTEXT_RE.search(context):
        return "locations"
    if PERSON_CONTEXT_RE.search(context):
        # Generic personal name pattern: 2-4 capitalized words, allowing common particles.
        words = c.split()
        if 2 <= len(words) <= 5:
            return "people"
    # Without strong context, keep as organization/product suggestion rather than person.
    if c.isupper() or any(ch.isdigit() for ch in c):
        return "methods_tools_or_products"
    return "organizations"

def is_bad_entity(text: str) -> bool:
    t = clean_term(text)
    tl = t.lower()
    if not t or len(t) < 2:
        return True
    if SECTION_NUMBER_RE.match(t) or DECIMAL_RE.match(t):
        return True
    if NUMBER_ONLY_RE.match(t) and not is_probable_date(t):
        return True
    if tl in STOPWORDS or tl in BAD_SINGLE_TOKENS:
        return True
    if len(t.split()) > 8:
        return True
    # Reject phrases that start or end with generic stopwords.
    parts = [p.lower().strip(".,;:()[]{}'") for p in t.split()]
    if parts and (parts[0] in STOPWORDS or parts[-1] in STOPWORDS):
        return True
    alpha_count = sum(ch.isalpha() for ch in t)
    if alpha_count < 2:
        return True
    return False

def add_entity(counters, category, text, confidence="low"):
    text = clean_term(text)
    if not category or is_bad_entity(text):
        return
    counters[category][text] += 1

def extract_dates_rule_based(text: str, counters):
    for pat in DATE_PATTERNS:
        for m in pat.finditer(text):
            add_entity(counters, "dates", m.group(0), "medium")

def extract_entities_rule_based(text: str) -> dict:
    counters = {cat: Counter() for cat in ENTITY_CATEGORIES}
    counters["groups_or_nationalities"] = Counter()
    extract_dates_rule_based(text, counters)

    for m in ORG_SUFFIX_RE.finditer(text):
        add_entity(counters, "organizations", m.group(1), "medium")

    for m in ACRONYM_RE.finditer(text):
        candidate = m.group(0)
        if is_probable_date(candidate):
            add_entity(counters, "dates", candidate, "medium")
        elif not is_bad_entity(candidate):
            ctx = nearby_context(text, m.start(), m.end())
            category = classify_capitalized_entity(candidate, ctx) or "methods_tools_or_products"
            add_entity(counters, category, candidate, "low")

    for m in CAPITALIZED_SEQUENCE_RE.finditer(text):
        candidate = m.group(0)
        ctx = nearby_context(text, m.start(), m.end())
        category = classify_capitalized_entity(candidate, ctx)
        if category:
            add_entity(counters, category, candidate, "low")

    return counters

def extract_entities_spacy(text: str) -> dict:
    counters = {cat: Counter() for cat in ENTITY_CATEGORIES}
    counters["groups_or_nationalities"] = Counter()
    if not SPACY_AVAILABLE or nlp is None:
        return counters

    # Process in chunks to avoid spaCy max length issues.
    max_len = 80000
    for i in range(0, len(text), max_len):
        part = text[i:i+max_len]
        doc = nlp(part)
        for ent in doc.ents:
            ent_text = clean_term(ent.text)
            if is_bad_entity(ent_text):
                continue
            category = SPACY_LABEL_MAP.get(ent.label_)
            if not category:
                continue
            if category == "dates" and not is_probable_date(ent_text):
                continue
            add_entity(counters, category, ent_text, "low")
    return counters

def merge_entity_counters(*counter_dicts):
    merged = {cat: Counter() for cat in ENTITY_CATEGORIES}
    merged["groups_or_nationalities"] = Counter()
    for d in counter_dicts:
        for cat, counter in d.items():
            if cat not in merged:
                merged[cat] = Counter()
            merged[cat].update(counter)
    return merged

def extract_entities_from_text(text: str) -> dict:
    rule_based = extract_entities_rule_based(text)
    spacy_based = extract_entities_spacy(text)
    return merge_entity_counters(rule_based, spacy_based)

def confidence_from_frequency(freq: int) -> str:
    if freq >= 5:
        return "high"
    if freq >= 2:
        return "medium"
    return "low"

def counters_to_suggestions(counters: dict, max_per_cat=25) -> dict:
    suggestions = {}
    for cat, counter in counters.items():
        items = []
        for text, freq in counter.most_common():
            if is_bad_entity(text):
                continue
            if cat == "dates" and not is_probable_date(text):
                continue
            items.append({
                "text": canonical_display(text),
                "frequency": freq,
                "confidence": confidence_from_frequency(freq),
                "note": "generic_suggestion_only_not_verified_metadata"
            })
            if len(items) >= max_per_cat:
                break
        suggestions[cat] = items
    return suggestions

entity_counters = extract_entities_from_text(main_text)
entity_suggestions = counters_to_suggestions(entity_counters, max_per_cat=30)
print("entity categories:", list(entity_suggestions.keys()))
for cat in ENTITY_CATEGORIES:
    print(cat, len(entity_suggestions.get(cat, [])), entity_suggestions.get(cat, [])[:5])


entity categories: ['people', 'organizations', 'locations', 'dates', 'methods_tools_or_products', 'documents_or_standards', 'events', 'groups_or_nationalities']
people 26 [{'text': 'Roy', 'frequency': 5, 'confidence': 'high', 'note': 'generic_suggestion_only_not_verified_metadata'}, {'text': 'Gramacki', 'frequency': 5, 'confidence': 'high', 'note': 'generic_suggestion_only_not_verified_metadata'}, {'text': 'Gramacki and Gramacki', 'frequency': 2, 'confidence': 'medium', 'note': 'generic_suggestion_only_not_verified_metadata'}, {'text': 'Kirkham et al', 'frequency': 1, 'confidence': 'low', 'note': 'generic_suggestion_only_not_verified_metadata'}, {'text': 'Maher et al', 'frequency': 1, 'confidence': 'low', 'note': 'generic_suggestion_only_not_verified_metadata'}]
organizations 30 [{'text': 'EEG', 'frequency': 87, 'confidence': 'high', 'note': 'generic_suggestion_only_not_verified_metadata'}, {'text': 'CNN', 'frequency': 24, 'confidence': 'high', 'note': 'generic_suggestion_only_not_veri

In [58]:

# ---------------------------------------------------------------------
# Chunk-level NLP
# ---------------------------------------------------------------------
def build_chunk_keywords(text: str, top_n=15):
    counts = extract_candidate_ngrams(text, 1, 4)
    items = []
    for term, freq in counts.items():
        if freq < 2 and len(term.split()) < 2:
            continue
        length_bonus = 1.0 + min(len(term.split()) - 1, 3) * 0.3
        score = freq * length_bonus
        items.append({
            "term": canonical_display(term),
            "frequency": freq,
            "score": round(score, 3)
        })
    items.sort(key=lambda x: (x["score"], len(x["term"].split()), x["frequency"]), reverse=True)
    return collapse_keyword_variants(items)[:top_n]

def build_chunk_nlp(chunks):
    out = []
    for ch in chunks:
        text = ch.get("text", "")
        counters = extract_entities_from_text(text)
        out.append({
            "chunk_id": ch.get("chunk_id", ""),
            "source_section_id": ch.get("source_section_id", ""),
            "source_section_heading": ch.get("source_section_heading", ""),
            "word_count": ch.get("word_count", len(text.split())),
            "keyword_suggestions": build_chunk_keywords(text, top_n=15),
            "entity_suggestions": counters_to_suggestions(counters, max_per_cat=10)
        })
    return out

chunk_nlp = build_chunk_nlp(chunks)
print("chunk_nlp records:", len(chunk_nlp))


chunk_nlp records: 57


In [59]:

# ---------------------------------------------------------------------
# Save outputs
# ---------------------------------------------------------------------
entity_count_total = sum(len(v) for v in entity_suggestions.values())

silver_nlp_output = {
    "document_id": DOCUMENT_ID,
    "original_file": silver.get("original_file", ""),
    "detected_language": detected_language,
    "processing_layer": "silver_nlp",
    "processing_version": "silver_nlp_local_generic_v5_no_document_specific_rules",
    "created_at": datetime.now().isoformat(),
    "source_silver_version": silver.get("processing_version", "unknown"),
    "local_execution_note": "This layer runs locally and does not call Ollama. It is compatible with any later Gold/Gold Meta Ollama model because the output schema is model-independent.",
    "genericity_note": "No document-specific terms, persons, organizations, locations, or project names are hardcoded. Suggestions are generated from the input document itself using generic NL/EN rules and optional local spaCy.",
    "important_note": "All entities and keywords in this layer are suggestions only. They are intended as hints for Gold and Gold Meta, not as final metadata.",
    "statistics": {
        "main_text_words": len(main_text.split()),
        "section_count": len(sections),
        "chunk_count": len(chunks),
        "keyword_suggestion_count": len(keyword_suggestions),
        "entity_suggestion_count_total": entity_count_total,
        "spacy_available": SPACY_AVAILABLE,
        "spacy_model": spacy_model_name,
    },
    "titlepage_candidates_from_silver": titlepage_candidates,
    "keyword_suggestions": keyword_suggestions,
    "entity_suggestions": entity_suggestions,
    "chunk_nlp": chunk_nlp,
}

paths = {
    "silver_nlp_json": OUT_DIR / f"{DOCUMENT_ID}_silver_nlp.json",
    "keyword_suggestions_json": OUT_DIR / f"{DOCUMENT_ID}_keyword_suggestions.json",
    "entity_suggestions_json": OUT_DIR / f"{DOCUMENT_ID}_entity_suggestions.json",
    "chunk_nlp_jsonl": OUT_DIR / f"{DOCUMENT_ID}_chunk_nlp.jsonl",
}

with open(paths["silver_nlp_json"], "w", encoding="utf-8") as f:
    json.dump(silver_nlp_output, f, ensure_ascii=False, indent=4)

with open(paths["keyword_suggestions_json"], "w", encoding="utf-8") as f:
    json.dump({
        "document_id": DOCUMENT_ID,
        "important_note": silver_nlp_output["important_note"],
        "keyword_suggestions": keyword_suggestions,
    }, f, ensure_ascii=False, indent=4)

with open(paths["entity_suggestions_json"], "w", encoding="utf-8") as f:
    json.dump({
        "document_id": DOCUMENT_ID,
        "important_note": silver_nlp_output["important_note"],
        "entity_suggestions": entity_suggestions,
    }, f, ensure_ascii=False, indent=4)

with open(paths["chunk_nlp_jsonl"], "w", encoding="utf-8") as f:
    for rec in chunk_nlp:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

for name, path in paths.items():
    print(name, "->", path)


silver_nlp_json -> ..\..\Data\silver_nlp\doc_01_silver_nlp.json
keyword_suggestions_json -> ..\..\Data\silver_nlp\doc_01_keyword_suggestions.json
entity_suggestions_json -> ..\..\Data\silver_nlp\doc_01_entity_suggestions.json
chunk_nlp_jsonl -> ..\..\Data\silver_nlp\doc_01_chunk_nlp.jsonl


In [60]:

# ---------------------------------------------------------------------
# Quick preview
# ---------------------------------------------------------------------
print("Top keyword suggestions:")
for item in keyword_suggestions[:15]:
    print(f"{item['rank']:>2}. {item['term']} | freq={item['frequency']} | spread={item['section_spread']} | score={item['score']}")

print("\nEntity suggestion preview:")
for cat, items in entity_suggestions.items():
    if items:
        print(f"\n{cat}:")
        for item in items[:10]:
            print(f"- {item['text']} ({item['frequency']}, {item['confidence']})")


Top keyword suggestions:
 1. data | freq=221 | spread=43 | score=353.6
 2. model | freq=117 | spread=31 | score=180.632
 3. eeg | freq=95 | spread=29 | score=143.333
 4. it | freq=80 | spread=35 | score=128.0
 5. seizure | freq=77 | spread=29 | score=116.175
 6. seizures | freq=75 | spread=23 | score=105.263
 7. graduation thesis angélique huige | freq=31 | spread=30 | score=96.997
 8. research | freq=68 | spread=23 | score=95.439
 9. which | freq=54 | spread=30 | score=82.421
10. graduation thesis angélique | freq=31 | spread=30 | score=80.437
11. thesis angélique huige | freq=31 | spread=30 | score=80.437
12. learning | freq=59 | spread=17 | score=76.596
13. results | freq=49 | spread=24 | score=69.632
14. each | freq=47 | spread=22 | score=65.14
15. graduation thesis | freq=31 | spread=30 | score=63.876

Entity suggestion preview:

people:
- Roy (5, high)
- Gramacki (5, high)
- Gramacki and Gramacki (2, medium)
- Kirkham et al (1, low)
- Maher et al (1, low)
- Ahmad et al (1, low)
-